In [1]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
from scraper import fetch_website_contents
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
message = "Hello, this is my first ever message to you! Hi!" 

messages = [{'role': 'user', "content": message}]


openai = OpenAI()
response = openai.chat.completions.create(model="gpt-5-nano",
                                          messages=messages)

response.choices[0].message.content

'Hi there! Nice to meet you. I’m glad you reached out.\n\nI can help with a lot of things, from answering questions to drafting texts, brainstorming ideas, learning new topics, coding help, planning, and more. If you’re not sure where to start, a few quick ideas:\n\n- Explain a concept in simple terms (math, science, history, etc.)\n- Draft an email, message, or post\n- Brainstorm ideas for a project, story, or gift\n- Help with study plans or homework questions\n- Write or debug code\n- Plan a trip, event, or daily routine\n- Translate or summarize text\n\nWhat would you like to do today? And do you prefer short answers or more detailed explanations?'

### 1. Website Summariser

In [6]:
website = fetch_website_contents("https://www.bbc.co.uk") # scraper built using beautiful soup
print(bbc[:1000])


BBC - Home

BBC Homepage
Skip to content
Accessibility Help
Your account
Notifications
Home
News
Sport
Weather
iPlayer
Sounds
Bitesize
CBBC
CBeebies
Food
More menu
More menu
Search BBC
Home
News
Sport
Weather
iPlayer
Sounds
Bitesize
CBBC
CBeebies
Food
Close menu
Welcome to the BBC
BBC Homepage
Scott Mills sacked from BBC Radio 2 over 'personal conduct'
He's not hosted his show since Tuesday and the BBC has not yet given any more details about the allegations.
Attribution
Culture
Live
.
Trump threatens to 'obliterate' Iran's energy facilities if deal not reached 'shortly'
Attribution
World
Live
.
Drivers to learn details of compensation scheme for mis-sold car finance
Attribution
Business
'I dread the phone ringing': Inside the kennels responding to vicious XL bully attacks
Attribution
UK
Travelodge investigating more reports of strangers accessing rooms, says CEO
Attribution
England
Easter bank holiday weekend set to be busiest on roads in four years
Attribution
UK
Why is Nasa going ba

In [12]:
system_prompt =  """ 
You are an assistant that analyses the content of a website, and provides short summary whilst ignoring text
that is navaigation related.
Respond in markdown format, and include a heading for the summary.
"""

user_prompt_prefix = """ Here are the contents of the website. 
provide a short summary of this website in markdown. If it includes news or announcements then include these too.
"""


messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': user_prompt_prefix + website}
]


In [9]:
response = openai.chat.completions.create(model='gpt-5-nano', messages=messages)
response.choices[0].message.content

'# BBC Homepage Summary\n\n- The page is the BBC’s homepage featuring a broad mix of current news across categories such as World, UK, Business, Culture, and Sport, plus sections like Newsbeat and regional updates.\n- Notable recent headlines include:\n  - Culture: Scott Mills sacked from BBC Radio 2 over "personal conduct."\n  - World: Trump threatens to "obliterate" Iran’s energy facilities if a deal isn’t reached soon.\n  - Business: Drivers to learn details of compensation schemes for mis-sold car finance.\n  - UK: Inside the kennels responding to vicious XL bully attacks.\n  - UK (England): Easter bank holiday weekend set to be the busiest on roads in four years.\n  - US & Canada: Why NASA is going back to the Moon.\n  - Wales: Cardiff City damages claim over Emiliano Sala death dismissed.\n  - Newsbeat: Sophie Turner injury halts filming on Tomb Raider TV show.\n  - World/Middle East: Photos of a damaged US radar jet at a Saudi base; analysis of US–Iran tensions.\n  - Sport: De Z

In [14]:
#build into function

def messages_for(website):
    return [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': user_prompt_prefix + website}
]

#messages_for(website)

def summarise(url: str):
    try: 
        website = fetch_website_contents(url) #use function above to generate list dict of messages
        if not website:
            return "Error: No content retrieved from website"
        website = str(website)
    
    except Exception as e:
        return f"Error fetching website content: {e}"

    try:
        response = openai.chat.completions.create( #invoke model
            model='gpt-5-mini',
            messages = messages_for(website))#messages for returns list of dicts of messages
        if not response or not response.choices:
            return f"No response from LLM"
        
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generating summary: {e}"

In [18]:
#test

summarise('https://www.bbc.co.uk')

'# Summary\n\nThis is the BBC homepage featuring a mix of breaking news, politics, business, culture and sport headlines.\n\n- Top headlines\n  - Scott Mills has been sacked from BBC Radio 2 over "personal conduct"; he has not hosted since Tuesday and the BBC has given few details about the allegations.\n  - Donald Trump threatens to "obliterate" Iran\'s energy facilities if a deal is not reached "shortly" (World).\n  - Drivers are set to learn details of a compensation scheme for mis‑sold car finance (Business).\n  - Travelodge is investigating further reports that strangers accessed guests\' rooms, according to its CEO (England).\n  - Easter bank holiday expected to be the busiest on roads in four years (UK).\n\n- Middle‑east / energy coverage\n  - Ongoing Iran war coverage, including why the US is targeting Kharg Island and photos of a heavily damaged US radar jet at a Saudi base.\n  - Features on rising oil prices: countries’ responses (fuel rations, free buses) and a look back to 

In [19]:
def display_summary(url):
    summary = summarise(url)
    display(Markdown(summary))

display_summary('https://www.bbc.co.uk')

# Summary of BBC homepage

- Top announcement: BBC Radio 2 presenter Scott Mills has been sacked over unspecified “personal conduct”; he has not presented since Tuesday and details remain limited.  
- International / World: US issues strong warning to Iran — Trump (reportedly) threatens to “obliterate” Iran’s energy facilities if no deal is reached; coverage also includes why the US has Kharg Island in its sights and photos of a heavily damaged US radar jet at a Saudi base.  
- UK / Home: Reports on Travelodge investigating further incidents of strangers accessing rooms; Easter bank holiday expected to be the busiest on roads in four years; an inside story on kennels responding to XL bully attacks.  
- Business / Economy: Details on a compensation scheme for mis‑sold car finance and explainer pieces on rising oil prices, fuel rations, and comparisons with the 1970s oil crisis.  
- Culture / Features: Sophie Turner’s injury halts filming on a Tomb Raider TV show.  
- Sport: Key headlines include De Zerbi in advanced talks to become Spurs manager, Tuchel’s reaction to England withdrawals, and other football updates (e.g., Scotland squad changes, Cardiff City legal news).  
- Popular explainers and features: “Why is NASA going back to the Moon?” and other in‑depth/contextual pieces.

(These are the main stories and announcements visible on the homepage; navigation and menu items were excluded.)

### 2. Exercise

In [20]:
system_prompt2 = """
You are an email assistant. You read and summarise emails and identify actions that should be taken, if any. You should identify intent and urgency.
The summary should be short and concise and any actions should be listed as bullet points.
"""

user_prompt2 = """  
Here is the content of my email. Please summarise and generate any actions I should take (if any)
"""

In [21]:
def generate_messages(email):
    return [
        {'role': 'system', 'content':system_prompt2},
        {'role': 'system', 'content': system_prompt2 + email}
    ]

def summarise_email(email):
    email = email
    messages = generate_messages(email)
    response = openai.chat.completions.create(
        model = 'gpt-5-mini',
        messages = generate_messages(email)
    )
    return response.choices[0].message.content

def display_email_summary(email):
    summary = summarise_email(email)
    display(Markdown(summary))

email = """ Hello everyone,
 
Our records show that we are still missing your timesheet for week ending 3rd April 2026.
 
It’s important that all timesheets are submitted today on Time@IBM so we can maximise revenue for March.
This will also enable us to ensure costs are recorded in the correct accounting period before the month closes.
 
Please could you submit your outstanding timesheet as soon as possible today. If you have already done so, thank you and please disregard this message.
 
Thanks in advance …… Nazreen"""

display_email_summary(email)

Summary:
Reminder from Nazreen that timesheets for week ending 3 Apr 2026 are still missing. Submit today on Time@IBM so revenue and costs are recorded in the correct March accounting period before month close.

Intent: Request/reminder (administrative).  
Urgency: High — must be done today.

Actions:
- Submit your timesheet for week ending 3 Apr 2026 on Time@IBM today.  
- If you’ve already submitted, no action needed.  
- If you cannot submit or have issues, reply to Nazreen (or contact payroll/Time@IBM support) immediately.